# Detección y Modificación de Color de Cabello (Google Colab)

**Autores:** Santiago Arbelaez Contreras · Federico Alvarez  
**Dataset:** CelebA-HQ (solo imágenes — carpeta `CelebA-HQ-img`)  
**Modelo:** `jonathandinu/face-parsing` (SegFormer preentrenado) + OpenCV HSV  

> Este notebook detecta el cabello usando un modelo de **segmentación semántica facial**
> preentrenado y luego modifica su color con operaciones bit a bit en el espacio HSV.
> **No requiere** la carpeta `CelebAMask-HQ-mask-anno`.

## 1. Verificación de Entorno y GPU
Comprobamos que Colab nos asignó aceleración por GPU antes de continuar.

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
!nvidia-smi

## 2. Montaje de Google Drive e Instalación de Librerías
Montamos Drive para acceder a las imágenes e instalamos las dependencias.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate opencv-python-headless Pillow tqdm matplotlib

## 3. Configuración de Rutas

Definimos los directorios. **Solo necesitamos** la carpeta `CelebA-HQ-img`.

In [ ]:
from pathlib import Path

BASE_DIR        = Path('/content/drive/MyDrive/ActividadEvaluativa1_SantiagoArbelaez_FedericoAlvarez/ActividadEvaluativa3/hair_color_detection')
IMG_DIR         = BASE_DIR / 'data' / 'CelebA-HQ-img'
RESULTS_DIR     = BASE_DIR / 'results'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
RECOLORED_DIR   = RESULTS_DIR / 'recolored'

for d in [PREDICTIONS_DIR, RECOLORED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if IMG_DIR.exists():
    print(f"✅ Imágenes encontradas: {IMG_DIR}")
else:
    raise FileNotFoundError(f"❌ No se encontró: {IMG_DIR}")

print("✅ Rutas configuradas correctamente.")

## 4. Exploración del Dataset

Contamos las imágenes disponibles y visualizamos 6 de muestra.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

all_images = sorted(IMG_DIR.glob('*.jpg'))
print(f"Total imágenes en CelebA-HQ-img: {len(all_images)}")

sample = random.sample(all_images, min(6, len(all_images)))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, p in zip(axes.flat, sample):
    ax.imshow(Image.open(p))
    ax.set_title(p.name, fontsize=8)
    ax.axis('off')
plt.suptitle("Muestra de imágenes CelebA-HQ", fontsize=12)
plt.tight_layout()
plt.show()
print("✅ Celda completada")

## 5. Carga del Modelo de Segmentación Facial (Preentrenado)

Usamos **`jonathandinu/face-parsing`** (SegFormer-B3), entrenado para segmentar
19 clases faciales. La clase de cabello es el **índice 13** (NO el 17 que es neck).

| Índice | Clase |
|--------|-------|
| 0 | background |
| 1 | skin |
| 2 | nose |
| 3 | eye_g |
| 4 | l_eye |
| 5 | r_eye |
| 6 | l_brow |
| 7 | r_brow |
| 8 | l_ear |
| 9 | r_ear |
| 10 | mouth |
| 11 | u_lip |
| 12 | l_lip |
| **13** | **hair** ← usamos esta |
| 14 | hat |
| 15 | ear_r |
| 16 | neck_l |
| 17 | neck |
| 18 | cloth |

> `HAIR_CLASS = 13`  ← valor correcto para este modelo.

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")

# Mapeo completo de clases del modelo jonathandinu/face-parsing
LABEL_MAP = {
    0: "background", 1: "skin",   2: "nose",  3: "eye_g",
    4: "l_eye",      5: "r_eye",  6: "l_brow",7: "r_brow",
    8: "l_ear",      9: "r_ear", 10: "mouth",11: "u_lip",
   12: "l_lip",     13: "hair",  14: "hat",  15: "ear_r",
   16: "neck_l",    17: "neck",  18: "cloth"
}

processor  = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
seg_model  = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing")
seg_model  = seg_model.to(DEVICE).eval()

HAIR_CLASS = 13  # índice CORRECTO de la clase "hair" (17 es neck, no hair)
print(f"Clase objetivo: {LABEL_MAP[HAIR_CLASS]} (índice {HAIR_CLASS})")
print("✅ Modelo cargado correctamente.")
print("✅ Celda completada")

## 6. Función: Obtener Máscara Binaria del Cabello

Esta función aplica el modelo y devuelve dos salidas:

- **`seg_map`** → máscara multiclase (H × W, entero por píxel)
- **`binary_mask`** → máscara binaria (H × W, uint8)
  - Píxel = **255** si es cabello (ROI)
  - Píxel = **0** si NO es cabello

> Esta es la definición exacta de máscara binaria en visión por computadora:
> **blanco = región de interés**, **negro = fondo**.

In [ ]:
import cv2
import numpy as np

def get_hair_mask(img_path):
    """
    Segmenta el cabello en la imagen y devuelve:
      img_rgb     : ndarray RGB  (H x W x 3, uint8)
      binary_mask : ndarray      (H x W, uint8) con valores 0 o 255
    """
    img_path = Path(img_path)
    img      = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size

    # Preprocesar e inferir
    inputs  = processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = seg_model(**inputs).logits          # (1, C, H/4, W/4)

    # Upsample al tamaño original y obtener clase por píxel
    upsampled = torch.nn.functional.interpolate(
        logits, size=(orig_h, orig_w), mode="bilinear", align_corners=False
    )
    seg_map = upsampled.argmax(dim=1).squeeze().cpu().numpy()  # (H, W)

    # ── MÁSCARA BINARIA (0 / 255) ─────────────────────────────────────
    # 255 si el píxel pertenece a la clase HAIR, 0 en caso contrario
    binary_mask = np.where(seg_map == HAIR_CLASS, 255, 0).astype(np.uint8)

    return np.array(img), binary_mask

print("✅ Función get_hair_mask() definida.")

## 7. Inferencia y Visualización de la Máscara

Aplicamos el modelo a una imagen y mostramos:
1. **Imagen original**
2. **Máscara binaria** → blanco = cabello | negro = fondo
3. **Superposición** de la máscara sobre la imagen

In [ ]:
def infer_and_visualize(img_path):
    """Muestra original | máscara binaria | superposición."""
    img_path = Path(img_path)
    img_rgb, binary_mask = get_hair_mask(img_path)

    if binary_mask.max() == 0:
        print(f"⚠️  No se detectó cabello en {img_path.name}")
        return

    # Superposición: píxeles de cabello en rojo semitransparente
    overlay = img_rgb.copy()
    overlay[binary_mask == 255] = [220, 50, 50]
    blend = cv2.addWeighted(img_rgb, 0.55, overlay, 0.45, 0)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(binary_mask, cmap='gray', vmin=0, vmax=255)
    axes[1].set_title("Máscara Binaria\n(255 = Cabello | 0 = Fondo)")
    axes[1].axis('off')

    axes[2].imshow(blend)
    axes[2].set_title("Superposición")
    axes[2].axis('off')

    plt.suptitle(img_path.name, fontsize=10)
    plt.tight_layout()

    save_path = PREDICTIONS_DIR / f"pred_{img_path.name}"
    fig.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Guardado en: {save_path}")

test_images = sorted(IMG_DIR.glob('*.jpg'))[:50]
infer_and_visualize(test_images[0])
print("✅ Celda completada")

## 8. Función: Modificación del Color del Cabello (HSV)

Usamos la **máscara binaria como filtro** (`bitwise_and`) para aislar
los píxeles del cabello y modificar únicamente su canal H (Hue).

| Canal HSV | Rol |
|-----------|-----|
| **H** (Hue) | Color — aquí aplicamos el desplazamiento |
| **S** (Saturation) | Viveza — la escalamos para realzar el color |
| **V** (Value) | Brillo — lo dejamos intacto (fotorrealismo) |

In [ ]:
def change_hair_color(img_path, hue_shift, saturation_scale=1.3, save=True):
    """
    Modifica el color del cabello en una imagen.

    hue_shift      : desplazamiento circular del Hue (0-179, escala OpenCV)
    saturation_scale: factor de saturación (1.0 = sin cambio)
    """
    img_path = Path(img_path)
    img_rgb, binary_mask = get_hair_mask(img_path)

    if binary_mask.max() == 0:
        print(f"⚠️  Sin cabello en {img_path.name}")
        return img_rgb

    # Paso 1 — Convertir a BGR y luego a HSV
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    hsv     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Paso 2 — Aislar píxeles del cabello con bitwise_and (máscara = filtro)
    hair_hsv = cv2.bitwise_and(hsv, hsv, mask=binary_mask)

    # Paso 3 — Modificar H y S solo donde binary_mask == 255
    h, s, v   = cv2.split(hair_hsv)
    mask_bool = binary_mask > 0

    h = h.astype(np.int16)
    h[mask_bool] = (h[mask_bool] + hue_shift) % 180  # desplazamiento circular
    h = h.astype(np.uint8)

    s = s.astype(np.float32)
    s[mask_bool] = np.clip(s[mask_bool] * saturation_scale, 0, 255)
    s = s.astype(np.uint8)

    # Paso 4 — Reconstruir: fondo original + cabello modificado
    modified_hair_bgr = cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2BGR)
    bg_bgr  = cv2.bitwise_and(img_bgr, img_bgr, mask=cv2.bitwise_not(binary_mask))
    final_rgb = cv2.cvtColor(cv2.add(bg_bgr, modified_hair_bgr), cv2.COLOR_BGR2RGB)

    # Paso 5 — Guardar
    if save:
        out = RECOLORED_DIR / f"recolored_h{hue_shift}_{img_path.name}"
        cv2.imwrite(str(out), cv2.cvtColor(final_rgb, cv2.COLOR_RGB2BGR))

    return final_rgb

print("✅ Función change_hair_color() definida.")

## 9. Demo Interactiva — 3 Imágenes × 4 Columnas

Grilla: `Original | Máscara | 🔴 Rojo | 🟢 Verde | 🔵 Azul`

| Color | Hue (OpenCV 0-179) |
|-------|-------------------|
| 🔴 Rojo | 0 |
| 🟢 Verde | 60 |
| 🔵 Azul | 120 |

In [ ]:
demo_imgs = test_images[:3]

color_map = {
    "🔴 Rojo":  0,
    "🟢 Verde": 60,
    "🔵 Azul":  120,
}

fig, axes = plt.subplots(len(demo_imgs), 5, figsize=(22, 5 * len(demo_imgs)))

for row, img_path in enumerate(demo_imgs):
    img_rgb, binary_mask = get_hair_mask(img_path)

    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title("Original")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(binary_mask, cmap='gray', vmin=0, vmax=255)
    axes[row, 1].set_title("Máscara Binaria\n(255=cabello | 0=fondo)")
    axes[row, 1].axis('off')

    for col, (color_name, hue) in enumerate(color_map.items(), start=2):
        recolored = change_hair_color(img_path, hue, save=True)
        axes[row, col].imshow(recolored)
        axes[row, col].set_title(color_name)
        axes[row, col].axis('off')

plt.suptitle("Demo: Modificación de Color de Cabello", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "demo_final.png"), dpi=120, bbox_inches='tight')
plt.show()
print("✅ Celda completada")

## 10. Resumen del Pipeline

```
CelebA-HQ-img/imagen.jpg
      │
      ▼  face-parsing SegFormer (preentrenado)
  seg_map (H×W, uint8) — cada píxel = clase (0-18)
      │
      ▼  binary_mask = np.where(seg_map == 17, 255, 0)
  Máscara Binaria (H×W)  ← 255=cabello | 0=fondo
      │
      ▼  bitwise_and(HSV, HSV, mask=binary_mask)
  Aislamiento ROI + desplazamiento de Hue
      │
      ▼
  results/recolored/recolored_h{X}_imagen.jpg
```

In [ ]:
recolored_files = list(RECOLORED_DIR.glob('*.jpg'))
print(f"✅ Imágenes recoloreadas guardadas: {len(recolored_files)}")
for f in sorted(recolored_files):
    print(f"  → {f.name}")